# 01 — Baseline: không trọng số và balanced

Chạy độc lập bằng Run All sau khi cấu hình. Dùng chung `src/baf.py`; dữ liệu và kết quả nằm ngoài src. Train tháng0–5, validation6, không dự đoán tháng7. Giữ các notebook đã chạy trước đây trong project cũ.

LR mỗi cấu hình một lượt; LightGBM mỗi cấu hình ba seed, tổng8 lượt.

## 1. Môi trường và cấu hình

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = None  # Ví dụ: r"D:\UIT\DS317\BAF_project_clean"
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (path for path in [here, *here.parents] if (path / "src/baf.py").is_file()),
        None,
    )
)
if ROOT is None:
    raise FileNotFoundError("Mở project BAF_project_clean hoặc điền PROJECT_DIR.")

sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
import baf

if Path(baf.__file__).resolve() != (ROOT / "src/baf.py").resolve():
    raise RuntimeError(
        "Kernel đang giữ module của project khác. Restart Kernel trước khi Run All."
    )

print("Project:", ROOT)
print("Kernel:", sys.executable)
print(pd.Series(baf.package_versions()).to_string())


Project: D:\UIT\DS317\BAF_project
Kernel: c:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe
numpy            2.4.2
pandas           3.0.1
scipy           1.17.1
scikit-learn     1.7.2
psutil           7.2.2
lightgbm         4.7.0


In [2]:
DATA_PATH = ROOT / "data/raw/Base.csv"
previous_data = ROOT.parent / "BAF_project/data/raw/Base.csv"
if not DATA_PATH.is_file() and previous_data.is_file():
    DATA_PATH = previous_data

EXPECTED_ROWS = 1_000_000
DATASET_TAG = "BAF_Base"
SEEDS = [42, 317, 2026]
THREADS = 4
TREE_PARAMETERS = baf.TREE_PARAMETERS.copy()

METHODS = baf.BASELINE_METHODS.copy()
RUN_DIR = baf.new_directory(ROOT / "results/runs", "baseline")
print("Data:", DATA_PATH)
print("Methods:", METHODS)
print("Output:", RUN_DIR)


Data: D:\UIT\DS317\BAF_project\data\raw\Base.csv
Methods: ['lr_none', 'lr_balanced', 'lgbm_none', 'lgbm_balanced']
Output: D:\UIT\DS317\BAF_project\results\runs\baseline_20261027_094947_c40fbc6c


## 2. Kiểm tra dữ liệu và trọng số
Mọi phép fit tiền xử lý và tính trọng số chỉ dùng train.

In [4]:
frame = baf.load_data(DATA_PATH, expected_rows=EXPECTED_ROWS)
parts = baf.temporal_parts(frame)
display(pd.DataFrame(baf.profile(parts)).T)

weights_table = []
labels = parts["train"].fraud_bool.to_numpy()
for method in METHODS:
    weights = baf.sample_weights(labels, method)
    weights_table.append(
        {
            "method": method,
            "weight_negative": float(weights[labels == 0][0]),
            "weight_positive": float(weights[labels == 1][0]),
            "mean_weight": float(weights.mean()),
        }
    )
display(pd.DataFrame(weights_table))


,n,positive,months
train,794989,8151,"[0, 1, 2, 3, 4, 5]"
validation,108168,1450,[6]
test,96843,1428,[7]


,method,weight_negative,weight_positive,mean_weight
0,lr_none,1.00000,1.000000,1.0
1,lr_balanced,0.50518,48.766348,1.0
2,lgbm_none,1.00000,1.000000,1.0
3,lgbm_balanced,0.50518,48.766348,1.0


## 3. Huấn luyện
Lượt mới luôn có thư mục mới. Log, cấu hình, source snapshot, metric và dự đoán được ghi tự động; lỗi được giữ lại.

In [5]:
metrics = baf.train_classical(
    data_path=DATA_PATH,
    output=RUN_DIR,
    methods=METHODS,
    seeds=SEEDS,
    threads=THREADS,
    tree_parameters=TREE_PARAMETERS,
    frame=frame,
    expected_rows=EXPECTED_ROWS,
    dataset_tag=DATASET_TAG,
)
display(metrics)


{
  "methods": [
    "lr_none",
    "lr_balanced",
    "lgbm_none",
    "lgbm_balanced"
  ],
  "seeds": [
    42,
    317,
    2026
  ],
  "threads": 4,
  "tree_parameters": {
    "objective": "binary",
    "class_weight": null,
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 100,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "reg_lambda": 1.0,
    "verbosity": -1,
    "deterministic": true,
    "force_col_wise": true
  },
  "protocol": "temporal",
  "mode": "full",
  "evaluation_partitions": [
    "validation"
  ],
  "dataset_tag": "BAF_Base"
}
                 n positive              months
train       794989     8151  [0, 1, 2, 3, 4, 5]
validation  108168     1450                 [6]
test         96843     1428                 [7]
{"method": "lr_none", "seed": 42, "partition": "validation", "threshold": 0.03854593477449659, "fit_seconds": 13.33805859999984, "peak_rss_mib": 1860.31640625, "mean_weigh

,method,seed,partition,threshold,fit_seconds,peak_rss_mib,mean_weight,weight_negative,weight_positive,warnings,...,precision_frozen_threshold,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp
0,lr_none,42,validation,0.038546,13.338059,1860.316406,1.0,1.00000,1.000000,,...,0.122732,0.056042,0.004138,0.400000,0.980033,0.986567,101400,5318,706,744
1,lr_balanced,42,validation,0.788362,24.369720,1264.843750,1.0,0.50518,48.766348,,...,0.121062,0.056052,0.790345,0.055264,0.886881,0.816073,101389,5329,716,734
2,lgbm_none,42,validation,0.043906,28.407362,626.078125,1.0,1.00000,1.000000,,...,0.124092,0.056024,0.055862,0.532895,0.981334,0.986687,101410,5308,698,752
3,lgbm_none,317,validation,0.043320,25.768060,639.343750,1.0,1.00000,1.000000,,...,0.125021,0.056348,0.054483,0.512987,0.981272,0.986632,101385,5333,688,762
4,lgbm_none,2026,validation,0.043093,25.915081,629.144531,1.0,1.00000,1.000000,,...,0.124118,0.056311,0.053793,0.561151,0.981327,0.986752,101383,5335,694,756
5,lgbm_balanced,42,validation,0.808975,29.341600,630.777344,1.0,0.50518,48.766348,,...,0.125000,0.056135,0.806897,0.057843,0.890036,0.821232,101405,5313,691,759
6,lgbm_balanced,317,validation,0.806860,29.086819,638.042969,1.0,0.50518,48.766348,,...,0.125369,0.056338,0.804138,0.058957,0.892522,0.825318,101388,5330,686,764
7,lgbm_balanced,2026,validation,0.807528,28.185004,635.894531,1.0,0.50518,48.766348,,...,0.126467,0.055922,0.805517,0.058237,0.890976,0.822776,101434,5284,685,765


## 4. Kiểm chứng và tổng hợp
Recall dùng thang0–1. Std là độ lệch chuẩn qua seed; một lượt thì để NaN, không tự điền0. Điểm validation không phải đánh giá độc lập ngoài mẫu.

In [6]:
report = baf.audit_results([RUN_DIR])
display(report.summary)
display(report.prediction_audit)
report.summary.to_csv(RUN_DIR / "summary.csv", index=False)


,method,partition,n_runs,ap_mean,ap_std,recall_mean,recall_std,fit_seconds_mean
0,lgbm_balanced,validation,3,0.181438,0.000743,0.525977,0.002217,28.871141
1,lgbm_none,validation,3,0.180348,0.001549,0.521839,0.003471,26.696834
2,lr_balanced,validation,1,0.155413,NaN,0.506207,NaN,24.369720
3,lr_none,validation,1,0.155743,NaN,0.513103,NaN,13.338059


,method,seed,partition,sha256,verified
0,lr_none,42,validation,3eacf6b6c75e38505216b8f8add1c20405231a46c02b20...,True
1,lr_balanced,42,validation,b342051a9f5eb2e5140f4d54dfb330fbaaaae59f3c156f...,True
2,lgbm_none,42,validation,54881b96717fcdf19b1d9386e321e5ab3d99f7ef6bd205...,True
3,lgbm_none,317,validation,0a726d8abdf59fa95507e5e7ea9b45d1cd1812686f8341...,True
4,lgbm_none,2026,validation,662b9bca3af0ca1153bfeb0e1c0d2d886e41bf6bdb3fee...,True
5,lgbm_balanced,42,validation,e24a5b9a985bdc7bdb59e8835e4c3266df11bc829ee5b1...,True
6,lgbm_balanced,317,validation,6e9f943a27e0843ec14c6c83044befd75ab9c7bc789cc4...,True
7,lgbm_balanced,2026,validation,5bcb4b9c701f64199f9e40a67848c42ee1930fa35cda55...,True


## 5. Xuất proof trong một lần
Hàm đọc lại predictions/log và thực thi notebook kiểm chứng mới. Không phụ thuộc Ctrl+S hoặc tên notebook đang mở. Lưu notebook đang mở khi kết thúc để giữ output giao diện; đó là thao tác riêng với việc tạo proof.

In [7]:
archive = baf.export_proof(
    run_paths=[RUN_DIR],
    output_root=ROOT / "results/proofs",
)


Verified prediction files: 8
Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_095435_38da43e7.zip
